In [1]:
import cmath
import math

In [2]:
# Extracted S Parameteres from Datasheets @ 9.5GHz
s11 = cmath.rect(0.608, math.radians(34.757))
s21 = cmath.rect(1.603, math.radians(-81.484))
s12 = cmath.rect(0.151, math.radians(-52.987))
s22 = cmath.rect(0.238, math.radians(49.648))
f = 9.5e9

In [3]:
# Check Stability
delta = s11 * s22 - s12 * s21; mag_delta = abs(delta)
K = (1 - abs(s11)**2 - abs(s22)**2 + mag_delta**2) / (2 * abs(s12 * s21))
delta_cond = mag_delta < 1; K_cond = K > 1; stab_cond = delta_cond and K_cond

print(f"∆ = {round(mag_delta, 3)} ∠ {round(math.degrees(cmath.phase(delta)), 3)}°", "→", f"|∆| {'<' if delta_cond else '>'} 1")
print(f"k = {round(K, 3)}", "→", f"K {'>' if K_cond else '<'} 1")
print(f"Amplifier is {'Unconditionally' if stab_cond else 'conditionally'} Stable @ {f/1e9}GHz")

∆ = 0.366 ∠ 59.891° → |∆| < 1
k = 1.462 → K > 1
Amplifier is Unconditionally Stable @ 9.5GHz


In [4]:
# Calculate Max Transducer Gain
G = (abs(s21)/abs(s12)) * (K - math.sqrt(K**2 - 1))

print("Amplifier Maximum Transducer Gain =", round(G, 3), "=", f"{round(10 * math.log10(G), 3)}dB")

Amplifier Maximum Transducer Gain = 4.199 = 6.231dB


In [5]:
# Get Reflection Coefficients
B1 = 1 + abs(s11)**2 - abs(s22)**2 - mag_delta**2; B2 = 1 + abs(s22)**2 - abs(s11)**2 - mag_delta**2
C1 = s11 - delta * s22.conjugate(); C2 = s22 - delta * s11.conjugate()
gamma_ms = (B1 - math.sqrt(B1**2 - 4 * abs(C1)**2)) / (2 * C1)
gamma_ml = (B2 - math.sqrt(B2**2 - 4 * abs(C2)**2)) / (2 * C2)

print(f"Γ_s = {round(abs(gamma_ms), 3)} ∠ {round(math.degrees(cmath.phase(gamma_ms)), 3)}°")
print(f"Γ_l = {round(abs(gamma_ml), 3)} ∠ {round(math.degrees(cmath.phase(gamma_ml)), 3)}°")

Γ_s = 0.625 ∠ -38.669°
Γ_l = 0.185 ∠ -118.652°


In [11]:
# Design Matching Networks
def stub_match(gamma_l):

    Gamma = gamma_l; Gamma_L = Gamma.conjugate()

    z_L = (1 + Gamma_L) / (1 - Gamma_L); y_L = 1 / z_L
    g = y_L.real; b = y_L.imag

    if g == 1:
        t1 = -b / 2
        t2 = 0
    else:
        root = math.sqrt((1-g)**2 * b**2 + 4 * g * (1-g)**2)

    mag_G = abs(Gamma_L); phase_G = cmath.phase(Gamma_L)
    cos_phi = -mag_G; phi1 = math.acos(cos_phi); phi2 = -phi1

    d1 = ((phase_G - phi1) / (4 * math.pi)) % 0.5; d2 = ((phase_G - phi2) / (4 * math.pi)) % 0.5

    Gamma_d1 = cmath.rect(mag_G, phi1); y1 = (1 - Gamma_d1) / (1 + Gamma_d1); b1 = y1.imag
    Gamma_d2 = cmath.rect(mag_G, phi2); y2 = (1 - Gamma_d2) / (1 + Gamma_d2); b2 = y2.imag

    l1 = (math.atan(-b1) / (2 * math.pi)) % 0.5
    l2 = (math.atan(-b2) / (2 * math.pi)) % 0.5

    return [(d1, l1), (d2, l2)]

print("Input Match:")
for i, (d, l) in enumerate(stub_match(gamma_ms)):
    print(f"Solution {i+1}: d = {d:.3f} lambda, l = {l:.3f} lambda, total length = {(l+d):.3f} lambda")

print()

print("Output Match:")
for i, (d, l) in enumerate(stub_match(gamma_ml)):
    print(f"Solution {i+1}: d = {d:.3f} lambda, l = {l:.3f} lambda, total length = {(l+d):.3f} lambda")

Input Match:
Solution 1: d = 0.375 lambda, l = 0.161 lambda, total length = 0.536 lambda
Solution 2: d = 0.232 lambda, l = 0.339 lambda, total length = 0.571 lambda

Output Match:
Solution 1: d = 0.025 lambda, l = 0.057 lambda, total length = 0.082 lambda
Solution 2: d = 0.305 lambda, l = 0.443 lambda, total length = 0.747 lambda
